# Import necessary libraries

In [171]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import timedelta
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import xgboost as xgb
from sklearn.metrics import mean_squared_error
colo_pal = sns.color_palette()
plt.style.use('fivethirtyeight')
from math import sqrt

# pip install xgboost==1.6.2


### Data loading

In [172]:
# df = pd.read_csv('../data/ForecastDataset_from_2016_to_2025_for_SKUs_started_with_90.csv')
# Data loading
df = pd.read_csv('../data/ForecastDataset_from_2016_To_2024_for_all_SKUs.csv')

In [173]:
df.head()

,ProductFamily,Order_ID,OrderDate,SKU_Description,WareHouseQuantity,SKU,SKU_category,Channel,ShipToState,ShippingZip
0,AIRD,54522457,12/23/2016,Air Doctor AD3000,1.0,90AD01AD01,Cross Sell,WEB_Radial,CA,91403
1,AIRD,54526264,12/28/2016,Air Doctor AD3000,1.0,90AD01AD01,Cross Sell,WEB_Radial,CA,91326
2,AIRD,54536438,12/30/2016,Air Doctor AD3000,1.0,90AD01AD01,Cross Sell,WEB_Radial,DE,19806
3,AIRD,54536647,12/31/2016,Air Doctor AD3000,1.0,90AD01AD01,Cross Sell,WEB_Radial,CA,94903
4,AIRD,54537034,12/31/2016,Air Doctor AD3000,1.0,90AD01AD01,Cross Sell,WEB_Radial,MI,48310


In [174]:
# Check numeric columns variance
numeric_variance = df.var(numeric_only=True)   #Only numeric columns
print("Variance of numeric columns:\n", numeric_variance)

Variance of numeric columns:
 Order_ID             8.015243e+14
WareHouseQuantity    2.552609e-01
dtype: float64


In [175]:
df.shape

(1048575, 10)

In [176]:
df.isna().sum()

ProductFamily            0
Order_ID                 0
OrderDate                0
SKU_Description      54667
WareHouseQuantity     4955
SKU                  10630
SKU_category         59623
Channel              12431
ShipToState            240
ShippingZip              3
dtype: int64

In [177]:
df.nunique()

ProductFamily             1
Order_ID             763423
OrderDate              2648
SKU_Description          86
WareHouseQuantity        24
SKU                      93
SKU_category              9
Channel                  33
ShipToState              98
ShippingZip           50110
dtype: int64

In [178]:
# df.dropna(inplace=True)

In [179]:
df.isna().sum()

ProductFamily            0
Order_ID                 0
OrderDate                0
SKU_Description      54667
WareHouseQuantity     4955
SKU                  10630
SKU_category         59623
Channel              12431
ShipToState            240
ShippingZip              3
dtype: int64

In [180]:
df.shape

(1048575, 10)

### Take a subset of features (Because we want weekly forecast by sku)

In [181]:
df['OrderDate'] = pd.to_datetime(df['OrderDate'], format="%m/%d/%Y", errors='coerce')
df = df.dropna(subset=['OrderDate', 'SKU', 'WareHouseQuantity']).copy()

In [182]:
## Keep relevant cols
df = df[['OrderDate', 'SKU', 'WareHouseQuantity']].copy()


In [183]:
df

,OrderDate,SKU,WareHouseQuantity
0,2016-12-23,90AD01AD01,1.0
1,2016-12-28,90AD01AD01,1.0
2,2016-12-30,90AD01AD01,1.0
3,2016-12-31,90AD01AD01,1.0
4,2016-12-31,90AD01AD01,1.0
...,...,...,...
1048570,2024-04-29,10AD350CMPK1,2.0
1048571,2024-04-29,10AD350CMPK1,1.0
1048572,2024-04-29,10AD550CMPK1,1.0
1048573,2024-04-29,10AD350CMPK1,1.0


### Weekly aggregation per SKU

Example:

If OrderDate = 2023-09-07 (Thursday)
 - EndWeek = 2023-09-10 (Sunday)
 - StartWeek = 2023-09-04 (Monday)

So weekly periods will always be `Monday → Sunday.`

In [184]:
import pandas as pd

# Ensure OrderDate is datetime
df["OrderDate"] = pd.to_datetime(df["OrderDate"])

# WeekEnd = Sunday of that week
df["WeekEnd"] = df["OrderDate"] + pd.to_timedelta((6 - df["OrderDate"].dt.weekday) % 7, unit="D")

# WeekStart = Monday before that Sunday
df["WeekStart"] = df["WeekEnd"] - pd.Timedelta(days=6)

# Aggregate weekly per SKU
weekly = (
    df.groupby(["SKU", "WeekStart", "WeekEnd"], as_index=False)
      .agg(WareHouseQuantity=("WareHouseQuantity", "sum"))
)

print(weekly.head(40))


            SKU  WeekStart    WeekEnd  WareHouseQuantity
0    10AD01CV01 2023-09-04 2023-09-10                1.0
1    10AD01CV01 2024-01-01 2024-01-07                1.0
2   10AD01FNA02 2023-01-23 2023-01-29               24.0
3   10AD01FNA02 2023-01-30 2023-02-05               31.0
4   10AD01FNA02 2023-02-06 2023-02-12               26.0
5   10AD01FNA02 2023-02-13 2023-02-19               39.0
6   10AD01FNA02 2023-02-20 2023-02-26               22.0
7   10AD01FNA02 2023-02-27 2023-03-05                4.0
8   10AD01FNA02 2023-03-06 2023-03-12                8.0
9   10AD01FNA02 2023-03-13 2023-03-19               12.0
10  10AD01FNA02 2023-03-20 2023-03-26               48.0
11  10AD01FNA02 2023-03-27 2023-04-02               14.0
12  10AD01FNA02 2023-04-03 2023-04-09                7.0
13  10AD01FNA02 2023-04-10 2023-04-16                4.0
14  10AD01FNA02 2023-04-17 2023-04-23                9.0
15  10AD01FNA02 2023-04-24 2023-04-30                6.0
16  10AD01FNA02 2023-05-01 2023

In [185]:
weekly

,SKU,WeekStart,WeekEnd,WareHouseQuantity
0,10AD01CV01,2023-09-04,2023-09-10,1.0
1,10AD01CV01,2024-01-01,2024-01-07,1.0
2,10AD01FNA02,2023-01-23,2023-01-29,24.0
3,10AD01FNA02,2023-01-30,2023-02-05,31.0
4,10AD01FNA02,2023-02-06,2023-02-12,26.0
...,...,...,...,...
6770,BOGUS SKU,2019-08-05,2019-08-11,1.0
6771,BOGUS SKU,2019-08-12,2019-08-18,2.0
6772,BOGUS SKU,2020-11-30,2020-12-06,1.0
6773,BOGUS SKU,2021-04-19,2021-04-25,1.0


### SKU Activity → Forecastability

Filter only those skus which are forecastable

In [186]:
# 3) SKU Activity → Forecastability
# =========================
latest_date = df['OrderDate'].max()

sku_activity = df.groupby("SKU").agg(
    StartDate=("OrderDate", "min"),
    EndDate=("OrderDate", "max"),
    # TotalOrders=("Order_ID", "count"),
    TotalQuantity=("WareHouseQuantity", "sum")
).reset_index()

sku_activity["ActiveDays"] = (sku_activity["EndDate"] - sku_activity["StartDate"]).dt.days
sku_activity["Status"] = sku_activity["EndDate"].apply(
    lambda x: "Inactive" if (latest_date - x).days > 90 else "Active"
)
sku_activity["Forecastability"] = sku_activity.apply(
    lambda row: (
        "Forecastable" if (row["Status"] == "Active" and row["ActiveDays"] >= 365)
        else ("Not Forecastable" if row["Status"] == "Active" else "Inactive")
    ),
    axis=1
)

forecastable_skus = sku_activity.loc[sku_activity["Forecastability"] == "Forecastable", "SKU"].unique()
print(f"Forecastable SKUs: {len(forecastable_skus)}")

Forecastable SKUs: 40


### Feature Engineering (only for forecastable SKUs)

In [187]:
weekly_filtered = weekly[weekly['SKU'].isin(forecastable_skus)].copy()

In [188]:
weekly_filtered.head(60)

,SKU,WeekStart,WeekEnd,WareHouseQuantity
2,10AD01FNA02,2023-01-23,2023-01-29,24.0
3,10AD01FNA02,2023-01-30,2023-02-05,31.0
4,10AD01FNA02,2023-02-06,2023-02-12,26.0
5,10AD01FNA02,2023-02-13,2023-02-19,39.0
6,10AD01FNA02,2023-02-20,2023-02-26,22.0
7,10AD01FNA02,2023-02-27,2023-03-05,4.0
8,10AD01FNA02,2023-03-06,2023-03-12,8.0
9,10AD01FNA02,2023-03-13,2023-03-19,12.0
10,10AD01FNA02,2023-03-20,2023-03-26,48.0
11,10AD01FNA02,2023-03-27,2023-04-02,14.0


### Adding features

In [189]:
LAG_LIST    = [1,2,3,4,8,12,26,52]
ROLL_LIST   = [4,8,12]
MIN_HISTORY = 40

def add_features(g: pd.DataFrame) -> pd.DataFrame:
    g = g.sort_values('WeekEnd').reset_index(drop=True)
    # Lags
    for L in LAG_LIST:
        g[f'lag_{L}'] = g['WareHouseQuantity'].shift(L)
    # Rolling means (shift by 1 to avoid leakage)
    for W in ROLL_LIST:
        g[f'roll_mean_{W}'] = g['WareHouseQuantity'].shift(1).rolling(W).mean()
    # Calendar
    iso = g['WeekEnd'].dt.isocalendar()
    g['weekofyear'] = iso.week.astype(int)
    g['month']      = g['WeekEnd'].dt.month.astype(int)
    g['quarter']    = g['WeekEnd'].dt.quarter.astype(int)
    return g



In [190]:
weekly_feat = (
    weekly_filtered.groupby('SKU', group_keys=True)
                   .apply(add_features)
                   .reset_index(drop=True)
)

C:\Users\Nilam Wadhavane\AppData\Local\Temp\ipykernel_23544\594761024.py:3: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(add_features)


In [191]:
weekly_feat.head(40)

,SKU,WeekStart,WeekEnd,WareHouseQuantity,lag_1,lag_2,lag_3,lag_4,lag_8,lag_12,lag_26,lag_52,roll_mean_4,roll_mean_8,roll_mean_12,weekofyear,month,quarter
0,10AD01FNA02,2023-01-23,2023-01-29,24.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4,1,1
1,10AD01FNA02,2023-01-30,2023-02-05,31.0,24.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5,2,1
2,10AD01FNA02,2023-02-06,2023-02-12,26.0,31.0,24.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6,2,1
3,10AD01FNA02,2023-02-13,2023-02-19,39.0,26.0,31.0,24.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7,2,1
4,10AD01FNA02,2023-02-20,2023-02-26,22.0,39.0,26.0,31.0,24.0,NaN,NaN,NaN,NaN,30.00,NaN,NaN,8,2,1
5,10AD01FNA02,2023-02-27,2023-03-05,4.0,22.0,39.0,26.0,31.0,NaN,NaN,NaN,NaN,29.50,NaN,NaN,9,3,1
6,10AD01FNA02,2023-03-06,2023-03-12,8.0,4.0,22.0,39.0,26.0,NaN,NaN,NaN,NaN,22.75,NaN,NaN,10,3,1
7,10AD01FNA02,2023-03-13,2023-03-19,12.0,8.0,4.0,22.0,39.0,NaN,NaN,NaN,NaN,18.25,NaN,NaN,11,3,1
8,10AD01FNA02,2023-03-20,2023-03-26,48.0,12.0,8.0,4.0,22.0,24.0,NaN,NaN,NaN,11.50,20.750,NaN,12,3,1
9,10AD01FNA02,2023-03-27,2023-04-02,14.0,48.0,12.0,8.0,4.0,31.0,NaN,NaN,NaN,18.00,23.750,NaN,13,4,2


In [192]:
feature_cols = (
    [c for c in weekly_feat.columns if c.startswith('lag_') or c.startswith('roll_mean_')]
    + ['weekofyear','month','quarter']
)

In [193]:
feature_cols

['lag_1',
 'lag_2',
 'lag_3',
 'lag_4',
 'lag_8',
 'lag_12',
 'lag_26',
 'lag_52',
 'roll_mean_4',
 'roll_mean_8',
 'roll_mean_12',
 'weekofyear',
 'month',
 'quarter']

In [194]:
# Remove initial rows with NaNs (due to lags/rolls)
weekly_feat = weekly_feat.dropna(subset=feature_cols).reset_index(drop=True)

In [195]:
weekly_feat.nunique()

SKU                    30
WeekStart             318
WeekEnd               318
WareHouseQuantity     654
lag_1                 654
lag_2                 655
lag_3                 656
lag_4                 658
lag_8                 660
lag_12                664
lag_26                633
lag_52                579
roll_mean_4          1283
roll_mean_8          1663
roll_mean_12         1879
weekofyear             53
month                  12
quarter                 4
dtype: int64

In [197]:
weekly_feat

,SKU,WeekStart,WeekEnd,WareHouseQuantity,lag_1,lag_2,lag_3,lag_4,lag_8,lag_12,lag_26,lag_52,roll_mean_4,roll_mean_8,roll_mean_12,weekofyear,month,quarter
0,10AD01FNA02,2024-01-22,2024-01-28,4.0,11.0,14.0,17.0,12.0,52.0,10.0,12.0,24.0,13.50,18.375,24.333333,4,1,1
1,10AD01FNA02,2024-03-04,2024-03-10,2.0,4.0,11.0,14.0,17.0,15.0,40.0,14.0,31.0,11.50,12.375,23.833333,10,3,1
2,10AD01FNB02,2024-01-29,2024-02-04,7.0,6.0,6.0,13.0,18.0,16.0,29.0,16.0,16.0,10.75,10.625,18.250000,5,2,1
3,10AD01FNB02,2024-02-05,2024-02-11,3.0,7.0,6.0,6.0,13.0,10.0,26.0,15.0,30.0,8.00,9.500,16.416667,6,2,1
4,10AD01FNB02,2024-02-12,2024-02-18,18.0,3.0,7.0,6.0,6.0,9.0,41.0,12.0,11.0,5.50,8.625,14.500000,7,2,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2756,90AD550AD01,2024-04-01,2024-04-07,126.0,102.0,101.0,66.0,78.0,70.0,95.0,192.0,72.0,86.75,81.125,88.000000,14,4,2
2757,90AD550AD01,2024-04-08,2024-04-14,78.0,126.0,102.0,101.0,66.0,62.0,115.0,80.0,88.0,98.75,88.125,90.583333,15,4,2
2758,90AD550AD01,2024-04-15,2024-04-21,91.0,78.0,126.0,102.0,101.0,76.0,110.0,62.0,40.0,101.75,90.125,87.500000,16,4,2
2759,90AD550AD01,2024-04-22,2024-04-28,88.0,91.0,78.0,126.0,102.0,94.0,87.0,66.0,33.0,99.25,92.000,85.916667,17,4,2


### Model Building

Take all the data for training except last 8 weeks

So for each SKU:

- **Training set** = all weeks except the last 8 weeks.
- **Testing set** = the last 8 weeks only.

In [284]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error
from math import sqrt

HORIZON = 8
results_all = []

# 1) Split train/test across ALL SKUs
train = weekly_feat.groupby("SKU").apply(lambda g: g.iloc[:-HORIZON]).reset_index(drop=True)  # Groups the dataset per SKU (so each SKU’s history is handled separately).
test  = weekly_feat.groupby("SKU").apply(lambda g: g.iloc[-HORIZON:]).reset_index(drop=True)

X_train, y_train = train[feature_cols], train["WareHouseQuantity"]
X_test,  y_test  = test[feature_cols], test["WareHouseQuantity"]

# 2) Train ONE global model
global_model = XGBRegressor(
    n_estimators=500,
    max_depth=7,
    learning_rate=0.05,
    subsample=1.0,
    colsample_bytree=1.0,
    reg_lambda=5.0,
    random_state=42
)
global_model.fit(X_train, y_train)

# 3) Predict on last 8 weeks per SKU
y_pred = global_model.predict(X_test)
y_pred = np.clip(y_pred, a_min=0, a_max=None)  # avoid negatives

# 4) Store results in dataframe
# pred_df = test[["SKU","WeekEnd"]].copy()
# pred_df["Predicted_WareHouseQuantity"] = y_pred

# 5) Evaluate overall error
mae  = mean_absolute_error(y_test, y_pred)
rmse = sqrt(np.mean((y_test - y_pred)**2))
print(f"Global Model Performance: MAE={mae:.2f}, RMSE={rmse:.2f}")



C:\Users\Nilam Wadhavane\AppData\Local\Temp\ipykernel_23544\2392831645.py:9: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  train = weekly_feat.groupby("SKU").apply(lambda g: g.iloc[:-HORIZON]).reset_index(drop=True)  # Groups the dataset per SKU (so each SKU’s history is handled separately).
C:\Users\Nilam Wadhavane\AppData\Local\Temp\ipykernel_23544\2392831645.py:10: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  tes

Global Model Performance: MAE=37.34, RMSE=90.17


In [ ]:


# 4) Store results in dataframe
pred_df = test[["SKU","WeekEnd"]].copy()
pred_df["Actual_WareHouseQuantity"] = test["WareHouseQuantity"].values

# Convert predictions to integers
pred_df["Predicted_WareHouseQuantity"] = np.round(y_pred).astype(int)

# 6) Add extra evaluation columns
pred_df["Difference_Between_Predicted_And_Actual"] = (
    pred_df["Predicted_WareHouseQuantity"] - pred_df["Actual_WareHouseQuantity"]
)

pred_df["Percent_Difference"] = (
    (pred_df["Difference_Between_Predicted_And_Actual"] / pred_df["Actual_WareHouseQuantity"]) * 100
).round(2)

# Apply ceiling on tolerance bounds
pred_df["Minus_10_Percent"] = np.ceil(pred_df["Actual_WareHouseQuantity"] * 0.9).astype(int)
pred_df["Plus_10_Percent"]  = np.ceil(pred_df["Actual_WareHouseQuantity"] * 1.1).astype(int)

# Acceptable vs Out of Range
within_10 = (
    (pred_df["Predicted_WareHouseQuantity"] >= pred_df["Minus_10_Percent"]) &
    (pred_df["Predicted_WareHouseQuantity"] <= pred_df["Plus_10_Percent"])
)
pred_df["Within_10_Percent"] = np.where(within_10, "Acceptable", "Out of Range")

# 7) Business Accuracy Calculation
business_accuracy = (pred_df["Within_10_Percent"] == "Acceptable").mean() * 100
print(f"Business Accuracy (±10% rule): {business_accuracy:.2f}%")

pred_df.to_csv("Predictions.csv")

# Show final dataframe
pred_df


Business Accuracy (±10% rule): 21.49%


,SKU,WeekEnd,Actual_WareHouseQuantity,Predicted_WareHouseQuantity,Difference_Between_Predicted_And_Actual,Percent_Difference,Minus_10_Percent,Plus_10_Percent,Within_10_Percent
0,10AD01FNA02,2024-01-28,4.0,15,11.0,275.00,4,5,Out of Range
1,10AD01FNA02,2024-03-10,2.0,17,15.0,750.00,2,3,Out of Range
2,10AD01FNB02,2024-03-17,18.0,14,-4.0,-22.22,17,20,Out of Range
3,10AD01FNB02,2024-03-24,10.0,21,11.0,110.00,9,11,Out of Range
4,10AD01FNB02,2024-03-31,8.0,22,14.0,175.00,8,9,Out of Range
...,...,...,...,...,...,...,...,...,...
223,90AD550AD01,2024-04-07,126.0,124,-2.0,-1.59,114,139,Acceptable
224,90AD550AD01,2024-04-14,78.0,110,32.0,41.03,71,86,Out of Range
225,90AD550AD01,2024-04-21,91.0,81,-10.0,-10.99,82,101,Out of Range
226,90AD550AD01,2024-04-28,88.0,101,13.0,14.77,80,97,Out of Range


In [32]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error
from math import sqrt

HORIZON = 8
results_all = []

# 1) Split train/test across ALL SKUs
train = weekly_feat.groupby("SKU", group_keys=False).apply(lambda g: g.iloc[:-HORIZON]).reset_index(drop=True)  # Groups the dataset per SKU (so each SKU’s history is handled separately).
test  = weekly_feat.groupby("SKU", group_keys=False).apply(lambda g: g.iloc[-HORIZON:]).reset_index(drop=True)

X_train, y_train = train[feature_cols], train["WareHouseQuantity"]
X_test,  y_test  = test[feature_cols], test["WareHouseQuantity"]

# 2) Train ONE global model
global_model = XGBRegressor(
    n_estimators=800,
    max_depth=3,
    learning_rate=0.01,
    subsample=0.9,
    colsample_bytree=0.3,
    reg_lambda=0.3,
    min_child_weight=2,
    max_delta_step=1,
    random_state=42
)
global_model.fit(X_train, y_train)

# 3) Predict on last 8 weeks per SKU
y_pred = global_model.predict(X_test)
y_pred = np.clip(y_pred, a_min=0, a_max=None)  # avoid negatives

# 4) Store results in dataframe
# pred_df = test[["SKU","WeekEnd"]].copy()
# pred_df["Predicted_WareHouseQuantity"] = y_pred

# 5) Evaluate overall error
mae  = mean_absolute_error(y_test, y_pred)
rmse = sqrt(np.mean((y_test - y_pred)**2))
print(f"Global Model Performance: MAE={mae:.2f}, RMSE={rmse:.2f}")




C:\Users\Nilam Wadhavane\AppData\Local\Temp\ipykernel_23544\2096633828.py:9: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  train = weekly_feat.groupby("SKU", group_keys=False).apply(lambda g: g.iloc[:-HORIZON]).reset_index(drop=True)  # Groups the dataset per SKU (so each SKU’s history is handled separately).
C:\Users\Nilam Wadhavane\AppData\Local\Temp\ipykernel_23544\2096633828.py:10: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence t

Global Model Performance: MAE=83.46, RMSE=186.14


In [34]:


# 4) Store results in dataframe
pred_df = test[["SKU","WeekEnd"]].copy()
pred_df["Actual_WareHouseQuantity"] = test["WareHouseQuantity"].values

# Convert predictions to integers
pred_df["Predicted_WareHouseQuantity"] = np.round(y_pred).astype(int)

# 6) Add extra evaluation columns
pred_df["Difference_Between_Predicted_And_Actual"] = (
    pred_df["Predicted_WareHouseQuantity"] - pred_df["Actual_WareHouseQuantity"]
)

pred_df["Percent_Difference"] = (
    (pred_df["Difference_Between_Predicted_And_Actual"] / pred_df["Actual_WareHouseQuantity"]) * 100
).round(2)

# Apply ceiling on tolerance bounds
pred_df["Minus_10_Percent"] = np.ceil(pred_df["Actual_WareHouseQuantity"] * 0.9).astype(int)
pred_df["Plus_10_Percent"]  = np.ceil(pred_df["Actual_WareHouseQuantity"] * 1.1).astype(int)

# Acceptable vs Out of Range
within_10 = (
    (pred_df["Predicted_WareHouseQuantity"] >= pred_df["Minus_10_Percent"]) &
    (pred_df["Predicted_WareHouseQuantity"] <= pred_df["Plus_10_Percent"])
)
pred_df["Within_10_Percent"] = np.where(within_10, "Acceptable", "Out of Range")

# 7) Business Accuracy Calculation
business_accuracy = (pred_df["Within_10_Percent"] == "Acceptable").mean() * 100
print(f"Business Accuracy (±10% rule): {business_accuracy:.2f}%")

# Show final dataframe
pred_df


Business Accuracy (±10% rule): 7.46%


,SKU,WeekEnd,Actual_WareHouseQuantity,Predicted_WareHouseQuantity,Difference_Between_Predicted_And_Actual,Percent_Difference,Minus_10_Percent,Plus_10_Percent,Within_10_Percent
0,10AD01FNA02,2024-01-28,4.0,9,5.0,125.00,4,5,Out of Range
1,10AD01FNA02,2024-03-10,2.0,8,6.0,300.00,2,3,Out of Range
2,10AD01FNB02,2024-03-17,18.0,9,-9.0,-50.00,17,20,Out of Range
3,10AD01FNB02,2024-03-24,10.0,9,-1.0,-10.00,9,11,Acceptable
4,10AD01FNB02,2024-03-31,8.0,9,1.0,12.50,8,9,Acceptable
...,...,...,...,...,...,...,...,...,...
223,90AD550AD01,2024-04-07,126.0,9,-117.0,-92.86,114,139,Out of Range
224,90AD550AD01,2024-04-14,78.0,9,-69.0,-88.46,71,86,Out of Range
225,90AD550AD01,2024-04-21,91.0,9,-82.0,-90.11,82,101,Out of Range
226,90AD550AD01,2024-04-28,88.0,9,-79.0,-89.77,80,97,Out of Range


In [250]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

# ========== STEP 1: Build sequences ==========
seq_len = 8   # number of past weeks to look back
sku_data = []

for sku, g in weekly.groupby("SKU"):
    g = g.sort_values("WeekEnd")   # keep time order
    series = g["WareHouseQuantity"].values.astype(float)

    X, y = [], []
    for i in range(len(series) - seq_len):
        X.append(series[i:i+seq_len])
        y.append(series[i+seq_len])

    if len(X) > 0:   # only keep SKUs with enough history
        X = np.array(X).reshape(-1, seq_len)   # (samples, seq_len)
        y = np.array(y)
        sku_data.append((X, y))

# Combine across SKUs
X_all = np.concatenate([d[0] for d in sku_data], axis=0)
y_all = np.concatenate([d[1] for d in sku_data], axis=0)

# Add feature dimension for LSTM
X_all = np.expand_dims(X_all, axis=-1)   # (samples, seq_len, 1)

print("Final shapes:", X_all.shape, y_all.shape)


# ========== STEP 2: Train / Test Split ==========
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.2, shuffle=True, random_state=42
)


# ========== STEP 3: Build LSTM Model ==========
model = Sequential([
    LSTM(64, input_shape=(seq_len, 1), return_sequences=False),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dropout(0.1),
    Dense(1)   # regression output
])

model.compile(optimizer='adam', loss='mse', metrics=['mae'])
model.summary()


# ========== STEP 4: Train ==========
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=20,
    batch_size=32,
    verbose=1
)


# ========== STEP 5: Evaluate ==========
test_loss, test_mae = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Loss (MSE): {test_loss:.4f}, Test MAE: {test_mae:.4f}")


Final shapes: (6123, 8, 1) (6123,)


c:\forecasting\workspace2\forecasting\.venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_1 (LSTM)                   │ (None, 64)             │        16,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 19,009 (74.25 KB)

 Trainable params: 19,009 (74.25 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
154/154 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 225373.6875 - mae: 184.1716 - val_loss: 198408.7812 - val_mae: 149.8739
Epoch 2/20
154/154 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 195609.2812 - mae: 150.4278 - val_loss: 171179.5469 - val_mae: 123.8627
Epoch 3/20
154/154 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 166087.8906 - mae: 127.1325 - val_loss: 147365.3281 - val_mae: 105.4743
Epoch 4/20
154/154 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 142011.9219 - mae: 115.2424 - val_loss: 132777.4375 - val_mae: 98.1595
Epoch 5/20
154/154 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 133173.8438 - mae: 110.9144 - val_loss: 127515.2812 - val_mae: 95.1753
Epoch 6/20
154/154 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 119594.9531 - mae: 105.4256 - val_loss: 116884.2188 - val_mae: 93.3025
Epoch 7/20
154/154 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 108291.4922 - mae: 101.1167 - val_loss: 111346.1953 - val_mae: 93.3197
Epoch 8/20
154/154 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 103597.9453 - mae: 9